In [1]:
"""
will_rg_core.py -- SymPy formalization of the zero-parameter core of
WILL Relational Geometry, Part I (WILL_RG_I).

Purpose
-------
Every boxed identity of WILL_RG_I is re-expressed as a symbolic residual that
must vanish *identically*, and each check records:

  * which symbols it consumed        -> provenance / channel-independence audit
  * which numeric constants appear   -> zero-parameter audit
  * the simplified residual          -> PASS iff residual == 0

Nothing here is fitted. A check is only allowed to consume symbols drawn from
the declared registers below. If a derivation needed a quantity outside those
registers, the audit flags it as SMUGGLED -- that is the mechanized version of
Principle (Epistemic Hygiene).

Registers
---------
PRIMITIVE  : carrier angles, amplitudes, phases, ledger-per-DOF
OBSERVABLE : spectroscopic shifts and light-times (operationally measurable)
SCALE      : unit-conversion labels (c, G, M, E0, m, r, Rs, a, D, M_mult)
COORD      : posited coordinate inflation (dx, dy, dz, dt, dtau) -- used ONLY
             in the reductio checks, where their appearance is the point.

Run:  python will_rg_core.py
"""

from __future__ import annotations

import sympy as sp

# ----------------------------------------------------------------------------
# Symbol registers
# ----------------------------------------------------------------------------

# --- PRIMITIVE: carriers S^1 (kinematic, 1 DOF) and S^2 (potential, 2 DOF)
th1, th2 = sp.symbols("theta_1 theta_2", real=True)
b, bY = sp.symbols("beta beta_Y", nonnegative=True)     # amplitude / phase on S^1
k, kX = sp.symbols("kappa kappa_X", nonnegative=True)   # amplitude / phase on S^2
ell = sp.symbols("ell", positive=True)                  # active ledger per quadratic DOF
eps = sp.symbols("varepsilon", positive=True)           # Taylor bookkeeping parameter

# --- OBSERVABLE: spectroscopy + light-times
zb, zk = sp.symbols("z_beta z_kappa", nonnegative=True)
tau = sp.symbols("tau", positive=True)
Zsys, ZA = sp.symbols("Z_sys Z_A", positive=True)
tA, dtA, td = sp.symbols("t_A Delta_t_A t_d", positive=True)

# --- SCALE: unit-conversion labels (never dynamical content)
c, G, Mass = sp.symbols("c G M", positive=True)
E0, m0 = sp.symbols("E_0 m_0", positive=True)
r, Rs, a = sp.symbols("r R_s a", positive=True)
rho, rho_max = sp.symbols("rho rho_max", positive=True)
alpha, n = sp.symbols("alpha n", positive=True)
D, Mult = sp.symbols("D M_mult", positive=True)
ratio = sp.symbols("varrho", positive=True)             # R_E / r_GPS (order unity)

# --- COORD: the posited container (reductio checks only)
dx, dy, dz, dt, dtau = sp.symbols("dx dy dz dt dtau", positive=True)

# Two-point states A, B
bA, bB, bYA, bYB = sp.symbols("beta_A beta_B beta_YA beta_YB", nonnegative=True)
kA, kB, kXA, kXB = sp.symbols("kappa_A kappa_B kappa_XA kappa_XB", nonnegative=True)

# local-but-legitimate symbols used inside individual checks
bAC_s = sp.Symbol("beta_AC", nonnegative=True)
kAC_s, kd_s = sp.symbols("kappa_AC kappa_d", nonnegative=True)
kE2_s, bE2_s = sp.symbols("kappa_E2 beta_E2", positive=True)
kXp_s = sp.Symbol("kappa_Xp", positive=True)
tauAC_s = sp.Symbol("tau_AC", positive=True)
RsA_s = sp.Symbol("R_sA", positive=True)
ro_s, to_s, v_s = sp.symbols("r_o t_o v", positive=True)

REGISTERS = {
    "PRIMITIVE": {th1, th2, b, bY, k, kX, ell, eps,
                  bA, bB, bYA, bYB, kA, kB, kXA, kXB,
                  bAC_s, kAC_s, kd_s, kE2_s, bE2_s, kXp_s},
    "OBSERVABLE": {zb, zk, tau, Zsys, ZA, tA, dtA, td, tauAC_s},
    "SCALE": {c, G, Mass, E0, m0, r, Rs, a, rho, rho_max, alpha, n, D, Mult, ratio,
              RsA_s, ro_s, to_s, v_s},
    "COORD": {dx, dy, dz, dt, dtau},
}
ALLOWED = set().union(*REGISTERS.values())


def register_of(sym) -> str:
    for name, members in REGISTERS.items():
        if sym in members:
            return name
    return "SMUGGLED"


# ----------------------------------------------------------------------------
# Substitution dictionaries that encode the *definitions* of the framework.
# These are the only bridges between symbols; each is traceable to WILL_RG_I.
# ----------------------------------------------------------------------------

TRIG = {b: sp.cos(th1), bY: sp.sin(th1), k: sp.sin(th2), kX: sp.cos(th2)}
PHASE = {bY: sp.sqrt(1 - b**2), kX: sp.sqrt(1 - k**2)}

# Energies (Thm Invariant Projection of Rest Energy; Lemma Unified Relational Scaling)
E_beta = E0 / bY                     # kinematic carrier energy
E_kappa = E0 / kX                    # potential carrier energy
E_total = E0 * kX / bY               # relational total energy
p_beta = E_beta * b                  # horizontal projection = momentum   (c = 1)
p_kappa = (E0 / c) * (k / kX)        # potential analogue of momentum


# ----------------------------------------------------------------------------
# Check harness
# ----------------------------------------------------------------------------

RESULTS: list[dict] = []


def record(cid, layer, claim, ref, residual, note="", extra_syms=frozenset(),
           expect="PASS"):
    """Simplify a residual that must vanish identically, and audit its content."""
    res = sp.simplify(sp.together(sp.expand(residual)))
    if res != 0:
        res = sp.simplify(sp.radsimp(res))
    syms = set(residual.free_symbols) | set(extra_syms)
    smuggled = sorted((str(s) for s in syms if register_of(s) == "SMUGGLED"))
    nums = sorted(
        {sp.nsimplify(x) for x in residual.atoms(sp.Number) if x not in (0, 1, -1)},
        key=lambda q: abs(float(q)),
    )
    status = "PASS" if res == 0 else "FAIL"
    RESULTS.append(dict(
        id=cid,
        layer=layer,
        claim=claim,
        ref=ref,
        status=status,
        expect=expect,
        verdict="OK" if status == expect else "PROBLEM",
        residual=sp.sstr(res),
        symbols=" ".join(sorted(str(s) for s in syms)),
        registers=" ".join(sorted({register_of(s) for s in syms})),
        constants=" ".join(sp.sstr(q) for q in nums),
        smuggled=" ".join(smuggled),
        note=note,
    ))
    return res


def record_value(cid, layer, claim, ref, got, expected, note=""):
    """Record a check whose content is 'this expression equals that expression'."""
    return record(cid, layer, claim, ref, sp.sympify(got) - sp.sympify(expected), note)


# ============================================================================
# LAYER 1 -- Carrier closure (Thm Conservation of Relation)
# ============================================================================

def layer1():
    L = "1 carriers"
    record("C1.1", L, "S^1 closure beta^2 + beta_Y^2 = 1 under beta=cos(th1), beta_Y=sin(th1)",
           "Thm carriers (a); Thm conservation",
           (b**2 + bY**2 - 1).subs(TRIG),
           "Closure is the Pythagorean identity of the unit circle: no parameter.")

    record("C1.2", L, "S^2 closure kappa^2 + kappa_X^2 = 1 under kappa=sin(th2), kappa_X=cos(th2)",
           "Thm carriers (b); Thm conservation",
           (k**2 + kX**2 - 1).subs(TRIG),
           "Meridional great-circle section of S^2; ledger normalised to unity.")

    record("C1.3", L, "Phase is fixed by amplitude: beta_Y = sqrt(1-beta^2)",
           "Sec kinetic",
           (bY**2 - (1 - b**2)).subs(PHASE))

    record("C1.4", L, "Phase is fixed by amplitude: kappa_X = sqrt(1-kappa^2)",
           "Sec potential",
           (kX**2 - (1 - k**2)).subs(PHASE))

    # The unit radius is the only normalisation: rescaling it would introduce a parameter.
    R = sp.symbols("R", positive=True)
    record("C1.5", L, "Any carrier radius R != 1 reintroduces a free parameter (must fail)",
           "Thm carriers; Pr epistemic",
           (b**2 + bY**2 - R**2).subs(TRIG),
           "NEGATIVE CONTROL: residual 1-R^2 is the smuggled scale; vanishes only at R=1.",
           extra_syms={R}, expect="FAIL")


# ============================================================================
# LAYER 2 -- Spectroscopic inversion (operational measurability)
# ============================================================================

def layer2():
    L = "2 spectroscopy"
    record("C2.1", L, "kappa^2 = 1 - 1/(1+z_kappa)^2  from  kappa_X = 1/(1+z_kappa)",
           "Thm Spectroscopic Phase Shift",
           (1 - kX**2).subs(kX, 1 / (1 + zk)) - (1 - 1 / (1 + zk) ** 2))

    record("C2.2", L, "beta^2 = 1 - 1/(1+z_beta)^2  from  beta_Y = 1/(1+z_beta)",
           "Thm Kinematic Phase Shift",
           (1 - bY**2).subs(bY, 1 / (1 + zb)) - (1 - 1 / (1 + zb) ** 2),
           "Transverse Doppler: observer's own amplitude vanishes against its local frame.")

    record("C2.3", L, "Round trip: 1+z_kappa = 1/kappa_X recovers kappa",
           "Thm Spectroscopic Phase Shift",
           sp.sqrt(1 - (1 / (1 + zk)) ** 2).subs(zk, 1 / sp.sqrt(1 - k**2) - 1) - k)

    record("C2.4", L, "tau = 1/[(1+z_kappa)(1+z_beta)] = kappa_X*beta_Y",
           "Thm Operational Measurability, step 1",
           (1 / ((1 + zk) * (1 + zb))).subs({zk: 1 / kX - 1, zb: 1 / bY - 1}) - kX * bY)

    record("C2.5", L, "tau^2 = 1 - (kappa^2+beta^2) + kappa^2 beta^2",
           "Thm Operational Measurability, step 2",
           ((1 - k**2) * (1 - b**2)) - (1 - (k**2 + b**2) + k**2 * b**2),
           "Multiplicative composition of the two phases -- source of the cross term.")

    record("C2.6", L, "Lorentz factor is the reciprocal phase: gamma = 1/beta_Y",
           "Summary after Thm restenergy",
           (1 / sp.sqrt(1 - b**2)) - (1 / bY).subs(PHASE))


# ============================================================================
# LAYER 3 -- Closure theorem, distance, bounds
# ============================================================================

def layer3():
    L = "3 closure"
    # kappa^2 = 2 beta^2 by eliminating the ledger-per-DOF ell. The '2' is a DOF count.
    ell_from_S1 = sp.solve(sp.Eq(b**2, 1 * ell), ell)[0]      # ell = beta^2  (1 DOF)
    k2_from_S2 = (2 * ell).subs(ell, ell_from_S1)              # kappa^2 = 2 ell (2 DOF)
    record("C3.1", L, "Closure kappa^2 = 2 beta^2 by eliminating ledger-per-DOF ell",
           "Thm Closure; Lem DOF-Indifference",
           k2_from_S2 - 2 * b**2,
           "beta^2 = 1*ell, kappa^2 = 2*ell -> kappa^2/beta^2 = 2 = dim S^2 / dim S^1.")

    record("C3.2", L, "Closure is equivalent to the DOF ratio itself",
           "Thm Closure",
           (k**2 / b**2).subs({k: sp.sqrt(2 * ell), b: sp.sqrt(ell)}) - 2)

    record("C3.3", L, "Spatial distance: r = R_s/kappa^2  <=>  kappa^2 = R_s/r",
           "Thm Inverse-Distance Potential Amplitude",
           (Rs / (Rs / r)) - r,
           "r is DEFINED as inverse amplitude per DOF (1/kappa * 1/kappa); no metric input.")

    record("C3.4", L, "Closure factor delta = kappa^2/(2 beta^2) equals 1 exactly on closure",
           "Def Closure Factor",
           (k**2 / (2 * b**2)).subs(k, sp.sqrt(2) * b) - 1)

    record("C3.5", L, "Eccentricity e = 2 beta^2/kappa^2 - 1 vanishes for a closed circular state",
           "Rem scope (ROM eccentricity)",
           (2 * b**2 / k**2 - 1).subs(k, sp.sqrt(2) * b),
           "e = 1/delta - 1, so circularity and closure are the same statement.")

    record("C3.6", L, "No singularity: beta_max^2=1 -> kappa_max^2=2 -> r_min = R_s/2",
           "Sec no_singularities",
           (Rs / (2 * b**2)).subs(b, 1) - Rs / 2)

    record("C3.7", L, "Static bound kappa^2 <= 1 forces beta^2 <= 1/2 for closed static states",
           "Thm Closure + S^2 additive closure",
           sp.solve(sp.Eq(2 * b**2, 1), b**2)[0] - sp.Rational(1, 2),
           "Derived corollary: circular closure saturates at beta = 1/sqrt(2), i.e. r = R_s.")

    # Channel-independence: the two derivations must not share variables.
    ch1 = (k**2 - Rs / r)        # potential <-> distance channel
    ch2 = (k**2 - 2 * b**2)      # potential <-> kinetic channel
    shared = ch1.free_symbols & ch2.free_symbols
    record("C3.8", L, "Channel independence: the 1/r channel and the factor-2 channel share only kappa",
           "Box Structural Independence of the Force Law and Virial Coefficient",
           sp.Integer(0) if shared == {k} else sp.Integer(1),
           f"shared symbols = {{{', '.join(sorted(str(s) for s in shared))}}}; "
           "r absent from closure, beta absent from the distance law.")

    record("C3.9", L, "Virial: closure kappa^2=2beta^2 <=> |V| = 2T with T=m v^2/2, V=-GMm/r",
           "Rem Geometric Origin of Physical Law",
           (k**2 * E0 / 2 - 2 * (b**2 * E0 / 2)).subs(k, sp.sqrt(2) * b),
           "|V| = (kappa^2/2)E_0 and T = (beta^2/2)E_0, so the virial coefficient IS the DOF ratio.")


# ============================================================================
# LAYER 4 -- Energy sector
# ============================================================================

def layer4():
    L = "4 energy"
    record("C4.1", L, "Invariant internal projection: E_beta * beta_Y = E_0",
           "Thm Invariant Projection of Rest Energy",
           E_beta * bY - E0)

    record("C4.2", L, "E_beta^2 = p_beta^2 + m^2 (c=1) with p_beta = E_beta*beta, m = E_0",
           "Cor Energy--Momentum Relation",
           sp.simplify((E_beta**2 - p_beta**2 - E0**2).subs(PHASE)),
           "Pure restatement of S^1 closure; mass enters only as the invariant E_0.")

    record("C4.3", L, "Trigonometric form: E_beta^2 = (cot(th1) E_0)^2 + E_0^2",
           "Rem Geometric Forms",
           ((E0 / bY) ** 2 - (b / bY * E0) ** 2 - E0**2).subs(TRIG))

    record("C4.4", L, "Restoring c: p_beta = gamma m v",
           "Rem Units sanity check",
           (E_beta * b / c).subs({E0: m0 * c**2, b: sp.Symbol("v", positive=True) / c})
           - (m0 * sp.Symbol("v", positive=True) / bY),
           extra_syms={sp.Symbol("v", positive=True)})

    record("C4.5", L, "Gravitational analogue: E_kappa^2 = (p_kappa c)^2 + (m c^2)^2",
           "Sec Gravitational Tangent Formulation",
           sp.simplify(((E_kappa) ** 2 - (p_kappa * c) ** 2 - E0**2).subs(PHASE)))

    record("C4.6", L, "Geometric equivalence: radial free fall beta=kappa gives p_beta = p_kappa",
           "Rem Ontological Status of p_kappa",
           sp.simplify((E0 * b / bY - E0 * k / kX).subs(PHASE).subs(k, b)))

    record("C4.7", L, "Equivalence principle: m_g = m_i = E_0/c^2 (single rest invariant)",
           "Lem Equivalence of Inertial and Gravitational Response",
           (p_beta.subs(E0, E0) / (b / bY) - p_kappa * c / (k / kX)) .subs(E0, E0),
           "Both momenta are the SAME E_0 scaled by different phase ratios.")

    record("C4.8", L, "Composition independence: channel decomposition E_0 = sum E_0^(a) cancels",
           "Rem Composition-Independence",
           sp.simplify(((E0 / 2 + E0 / 2) * kX / bY) - E0 * kX / bY),
           "Every internal channel scales by the identical phase ratio kappa_X/beta_Y.")

    record("C4.9", L, "Reciprocal duality at beta=0: E * E_kappa = E_0^2",
           "Sec energy quantities",
           sp.simplify((E_total * E_kappa).subs(bY, 1) - E0**2))

    record("C4.10", L, "Energy symmetry: Delta E_{A->B} + Delta E_{B->A} = 0",
           "Thm Energy Symmetry",
           (E0 * (kXB / bYB - kXA / bYA)) + (E0 * (kXA / bYA - kXB / bYB)),
           "Antisymmetry of a state function difference: identically zero, no parameter.")

    f = sp.Function("f")
    record("C4.10b", L, "Control: the same antisymmetry holds for an ARBITRARY state function",
           "Thm Energy Symmetry (scope probe)",
           (f(kXB, bYB) - f(kXA, bYA)) + (f(kXA, bYA) - f(kXB, bYB)),
           "So the zero-sum law alone constrains nothing; all physical content sits in "
           "the specific form E = E_0 kappa_X/beta_Y (Lem Unified Relational Scaling).")

    # Causal horizons as limits of the SAME expression.
    kXp = sp.Symbol("kappa_Xp", positive=True)   # interior of the carrier: 0 < kappa_X < 1
    limit_b = sp.limit(E_total.subs({kX: kXp, bY: sp.Symbol("x", positive=True)}),
                       sp.Symbol("x", positive=True), 0, "+")
    limit_k = sp.limit(E_total.subs(kX, sp.Symbol("y", positive=True)),
                       sp.Symbol("y", positive=True), 0, "+")
    record("C4.11", L, "Kinematic phase exhaustion beta_Y->0 gives E -> oo (speed of light)",
           "Thm Universal Rate of Change and the Horizon of Causality",
           sp.Integer(0) if limit_b == sp.oo else sp.Integer(1),
           f"limit = {sp.sstr(limit_b)}")
    record("C4.12", L, "Potential phase exhaustion kappa_X->0 gives E -> 0 (event horizon)",
           "Thm Universal Rate of Change and the Horizon of Causality",
           sp.simplify(limit_k),
           f"limit = {sp.sstr(limit_k)}")


# ============================================================================
# LAYER 5 -- Legacy translation (reductio ad absurdum checks)
# ============================================================================

def layer5():
    L = "5 legacy"
    # Taylor origin of the classical 1/2
    ratio_expr = sp.sqrt(1 - eps * k**2) / sp.sqrt(1 - eps * b**2)
    ser = sp.series(ratio_expr, eps, 0, 3).removeO()
    first = sp.expand(sp.simplify(ser.coeff(eps, 1)))
    record("C5.1", L, "First-order phase ratio: kappa_X/beta_Y ~ 1 - kappa^2/2 + beta^2/2",
           "Sec linearized relational limit",
           first - (-k**2 / 2 + b**2 / 2),
           "The classical factor 1/2 is the first Taylor coefficient -- not a postulate.")

    second = sp.expand(sp.simplify(ser.coeff(eps, 2)))
    record("C5.2", L, "Second order carries the cross term -kappa^2 beta^2/4",
           "Sec linearized relational limit",
           second.coeff(k**2 * b**2) - sp.Rational(-1, 4),
           f"second-order coefficient = {sp.sstr(second)}")

    # Two-point linearization
    lhs = ((1 - kB**2 / 2 + bB**2 / 2) - (1 - kA**2 / 2 + bA**2 / 2))
    record("C5.3", L, "Linearized two-point law: (kappa_A^2-kappa_B^2)/2 + (beta_B^2-beta_A^2)/2",
           "Eq linearized_two_point",
           lhs - (sp.Rational(1, 2) * (kA**2 - kB**2) + sp.Rational(1, 2) * (bB**2 - bA**2)),
           "The constant 1 cancels: only differences of squared amplitudes survive.")

    # Hamiltonian collapse: observer at infinity (kappa_A = beta_A = 0)
    v = sp.Symbol("v", positive=True)
    H_rel = (E0 * (sp.Rational(1, 2) * bB**2 - sp.Rational(1, 2) * kB**2)).subs(
        {bB: v / c, kB: sp.sqrt(2 * G * Mass / (r * c**2)), E0: m0 * c**2})
    record("C5.4", L, "Newtonian Hamiltonian H = m v^2/2 - GMm/r as the single-point collapse",
           "Sec Hamiltonian",
           sp.simplify(H_rel - (m0 * v**2 / 2 - G * Mass * m0 / r)),
           "Requires the operationally impossible frame at infinity (kappa_A=beta_A=0).",
           extra_syms={v})

    # Minkowski interval as inflation of S^1 closure
    inflated = ((dx**2 + dy**2 + dz**2) / (c**2 * dt**2) + (dtau / dt) ** 2 - 1) * c**2 * dt**2
    record("C5.5", L, "Minkowski interval = S^1 closure x (posited c^2 dt^2)",
           "Sec SR_interval",
           sp.expand(inflated) - sp.expand(dx**2 + dy**2 + dz**2 + c**2 * dtau**2 - c**2 * dt**2),
           "Four posits (container, xyz, autonomous t, scale c^2dt^2) added to beta^2+beta_Y^2=1.")

    # Schwarzschild g_tt as inflation of S^2 closure
    infl2 = ((dtau / dt) ** 2 + Rs / r - 1) * c**2 * dt**2
    record("C5.6", L, "Schwarzschild g_tt = S^2 closure x (posited c^2 dt^2)",
           "Sec GR_interval",
           sp.expand(infl2) - sp.expand(c**2 * dtau**2 - c**2 * (1 - Rs / r) * dt**2),
           "Same four posits; kappa^2 localized as R_s/r introduces r, G, M.")

    record("C5.7", L, "GR dictionary: kappa_X = sqrt(-g_tt) for static spacetimes",
           "Legacy Dictionary",
           sp.sqrt(1 - Rs / r) - kX.subs(kX, sp.sqrt(1 - k**2)).subs(k**2, Rs / r),
           "Pragmatic translation, not an ontological identity.")


# ============================================================================
# LAYER 6 -- Density, pressure, unified field equation
# ============================================================================

def layer6():
    L = "6 field"
    rho_expr = k**2 * c**2 / (8 * sp.pi * G * r**2)
    rho_max_expr = c**2 / (8 * sp.pi * G * r**2)

    record("C6.1", L, "Mass label from geometry: M = kappa^2 c^2 r /(2G) via R_s = 2GM/c^2",
           "Sec density",
           sp.solve(sp.Eq(k**2, 2 * G * Mass / (r * c**2)), Mass)[0] - k**2 * c**2 * r / (2 * G))

    record("C6.2", L, "Normalised identity kappa^2 = rho/rho_max (G and M cancel)",
           "Lem norm_id; Eq unified_field",
           sp.simplify(rho_expr / rho_max_expr - k**2),
           "S^2 surface normalisation 1/(4pi) applied to M/r^3; both G and M drop out.")

    record("C6.3", L, "Self-consistency forces n=3 and alpha=4pi in M = alpha r^n rho",
           "Sec Self-Consistency Requirement",
           sp.simplify((alpha * r ** (n - 2) / (8 * sp.pi) - r / 2).subs({n: 3, alpha: 4 * sp.pi})),
           "n is fixed by r-independence of M; alpha then follows. Note 4pi, not Newton's 4pi/3.")

    record("C6.4", L, "Closure of the loop: M = 4 pi r^3 rho reproduces M = kappa^2 c^2 r/(2G)",
           "Sec Self-Consistency Requirement",
           sp.simplify(4 * sp.pi * r**3 * rho_expr - k**2 * c**2 * r / (2 * G)))

    P = (c**4 / (8 * sp.pi * G)) * (1 / r) * sp.diff((Rs / r), r)
    record("C6.5", L, "Equation of state P = -rho c^2 from the radial balance relation",
           "Sec pressure",
           sp.simplify(P - (-rho_expr.subs(k**2, Rs / r) * c**2)),
           "d(kappa^2)/dr = -kappa^2/r drives the negative surface tension.")

    record("C6.6", L, "Saturation: P_max = -c^4/(8 pi G r^2) at kappa^2 = 1",
           "Sec pressure",
           sp.simplify((-rho_expr * c**2).subs(k, 1) - (-c**4 / (8 * sp.pi * G * r**2))))

    record("C6.7", L, "Vacuum field equation d(r kappa^2)/dr = 0 gives r kappa^2 = R_s",
           "Sec Field Equation and Matter Sources",
           sp.simplify(sp.diff(r * (Rs / r), r)),
           "The 1/r law is the vacuum solution of the accumulation equation.")

    record("C6.8", L, "Matter source: d(r kappa^2)/dr = 8 pi G r^2 rho_matter/c^2 is dimensionally closed",
           "Eq will_field_diff",
           sp.simplify((8 * sp.pi * G / c**2) * r**2 * rho_expr - k**2),
           "Substituting rho_matter = rho_field reduces the source term to kappa^2 itself.")

    record("C6.9", L, "Bounds: kappa^2 <= 2 implies rho <= 2 rho_max",
           "Sec no_singularities",
           sp.simplify((rho_expr / rho_max_expr).subs(k**2, 2) - 2))


# ============================================================================
# LAYER 7 -- Newton's rotating globes (fully operational inventory)
# ============================================================================

def layer7():
    L = "7 globes"
    bAC = sp.Symbol("beta_AC", nonnegative=True)
    kAC, kd = sp.symbols("kappa_AC kappa_d", nonnegative=True)
    tAC = sp.Symbol("tau_AC", positive=True)

    record("C7.1", L, "Total phase of the closed pair: tau_AC^2 = (1-2 beta^2)(1-beta^2)",
           "Sec Newton's Question Answered",
           (sp.sqrt(1 - 2 * bAC**2) * sp.sqrt(1 - bAC**2)) ** 2
           - (1 - 2 * bAC**2) * (1 - bAC**2),
           "kappa_X,tot uses kappa_tot^2 = 2 beta^2 from closure.",
           extra_syms={bAC})

    inv = sp.Rational(1, 4) * (3 - sp.sqrt(1 + 8 * tAC**2))
    record("C7.2", L, "Inversion beta^2 = (3 - sqrt(1+8 tau^2))/4 solves the closure quadratic",
           "Eq globes-answer",
           sp.simplify(((1 - 2 * inv) * (1 - inv) - tAC**2)),
           "Physical branch: the (3+sqrt) root gives beta^2=1 and is discarded.",
           extra_syms={tAC})

    record("C7.3", L, "At tau=0 the inversion saturates at beta^2 = 1/2 (r = R_s)",
           "Eq globes-answer",
           sp.limit(inv, tAC, 0) - sp.Rational(1, 2), extra_syms={tAC})

    # R_sA inverted from the two-station shift on globe A
    RsA = sp.Symbol("R_sA", positive=True)
    ZA_expr = sp.sqrt((1 - RsA / (c * (tA + dtA))) / (1 - RsA / (c * tA)))
    RsA_sol = sp.solve(sp.Eq(ZA**2, ZA_expr**2), RsA)[0]
    record("C7.4", L, "Inversion of the two-station shift gives R_sA(Z_A, t_A, Delta t_A)",
           "Eq globes-RsA",
           sp.simplify(RsA_sol - c * (ZA**2 - 1) / (ZA**2 / tA - 1 / (tA + dtA))),
           extra_syms={RsA})

    kAC_expr = sp.simplify(RsA_sol / (c * td / 2))
    record("C7.5", L, "kappa_AC^2 = R_sA/a with a = c t_d/2: the speed of light cancels",
           "Eq globes-kappaAC",
           sp.simplify(kAC_expr - 2 * (ZA**2 - 1) / (td * (ZA**2 / tA - 1 / (tA + dtA)))),
           "c does not appear in the result: two shifts and three light-times suffice.")

    record("C7.6", L, "Cord share: kappa_d^2 = 2 beta_AC^2 - kappa_AC^2",
           "Thm Restoring Closure",
           sp.solve(sp.Eq((kAC**2 + kd**2) / (2 * bAC**2), 1), kd**2)[0]
           - (2 * bAC**2 - kAC**2),
           "Quadratic additivity across channels is DOF-indifference, not an extra postulate.",
           extra_syms={bAC, kAC, kd})

    closed_form = (sp.Rational(1, 2) * (3 - sp.sqrt(1 + 8 / Zsys**2))
                   - 2 * (ZA**2 - 1) / (td * (ZA**2 / tA - 1 / (tA + dtA))))
    assembled = (2 * inv.subs(tAC, 1 / Zsys) - kAC_expr)
    record("C7.7", L, "Fully operational closed form for kappa_d^2 (two shifts, three light-times)",
           "Eq globes-kappad-closed",
           sp.simplify(assembled - closed_form))

    record("C7.8", L, "Classical tension is the cord's ledger share: T = E_0 kappa_d^2/d",
           "Sec Reductio ad Absurdum of the Classical Tension",
           sp.simplify((E0 * bAC**2 / a - E0 * kAC**2 / (2 * a))
                       - E0 * (2 * bAC**2 - kAC**2) / (2 * a)),
           "d = 2a, so T = E_0 kappa_d^2 / d. Force is not a primitive here.",
           extra_syms={bAC, kAC})

    record("C7.9", L, "Optional period label: T = pi t_d / beta_AC, N = pi/beta_AC",
           "Rem Optional period",
           sp.simplify((sp.pi * td / bAC) / td - sp.pi / bAC), extra_syms={bAC})


# ============================================================================
# LAYER 8 -- The W_ILL invariant
# ============================================================================

def layer8():
    L = "8 invariant"
    M_w = (b**2 / bY) * (c**2 * a / G)
    E_w = (k**2 / kX) * (c**4 * a / (2 * G))
    T_w = kX * (2 * G * m0 / (k**2 * c**3)) ** 2
    L_w = bY * (G * m0 / (b**2 * c**2)) ** 2

    W = sp.simplify(E_w * T_w / (M_w * L_w))
    record("C8.1", L, "W_ILL = E T/(M L) reduces to 2 beta^2/kappa^2 -- all constants cancel",
           "Sec willinvariant",
           sp.simplify(W - 2 * b**2 / k**2),
           f"reduced form = {sp.sstr(W)}; a, G, c, m_0 all cancel identically.")

    record("C8.2", L, "W_ILL = 1 is EXACTLY equivalent to the closure theorem kappa^2 = 2 beta^2",
           "Sec willinvariant",
           sp.simplify(W.subs(k, sp.sqrt(2) * b) - 1),
           "So W_ILL carries no content beyond closure: it is closure in dimensionful dress.")

    # Phase-normalised form
    r_o, t_o = sp.symbols("r_o t_o", positive=True)
    Wo = sp.simplify((E0 / kX * (kX * t_o**2)) / ((m0 / bY) * (bY * r_o**2)))
    record("C8.3", L, "Phase-normalised W_ILL = E_0 t_o^2/(m_0 r_o^2) = 1 given E_0=m_0c^2, r_o=c t_o",
           "Sec willinvariant (phase-normalised form)",
           sp.simplify(Wo.subs({E0: m0 * c**2, r_o: c * t_o}) - 1),
           "Phases cancel pairwise; the residual content is the light-time relation r = c t.",
           extra_syms={r_o, t_o})

    record("C8.4", L, "Sector coupling E_o/M_o = L_o/T_o",
           "Sec willinvariant",
           sp.simplify((E0 / kX) / (m0 / bY) - (bY * r_o**2) / (kX * t_o**2))
           .subs({E0: m0 * c**2, r_o: c * t_o}),
           extra_syms={r_o, t_o})


# ============================================================================
# LAYER 9 -- Earth/GPS: is 1PN GR the first-order Taylor coefficient of RG?
# ============================================================================

GPS = {}


def layer9():
    L = "9 gps"
    kE2, bE2 = sp.symbols("kappa_E2 beta_E2", positive=True)

    # exact RG shift coefficient; varrho = R_E/r_GPS is finite and NOT expanded
    delta_RG = 1 - sp.sqrt((1 - kE2) * (1 - bE2)) / sp.sqrt(
        (1 - kE2 * ratio) * (1 - kE2 * ratio / 2))

    scaled = delta_RG.subs({kE2: eps * kE2, bE2: eps * bE2})
    ser = sp.series(scaled, eps, 0, 3).removeO()
    d1 = sp.expand(sp.simplify(ser.coeff(eps, 1)))
    d2 = sp.expand(sp.simplify(ser.coeff(eps, 2)))

    # GR 1PN coefficient, rebuilt from its own definition
    kG2 = kE2 * ratio
    bG2 = kG2 / 2                       # circular orbital closure
    delta_GR = sp.expand((-kG2 / 2 + kE2 / 2) - (bG2 - bE2) / 2)

    record("C9.1", L, "GR 1PN coefficient equals kappa_E^2/2 + beta_E^2/2 - 3 kappa_E^2 varrho/4",
           "Eq delta-GR",
           delta_GR - (kE2 / 2 + bE2 / 2 - 3 * kE2 * ratio / 4),
           extra_syms={kE2, bE2})

    record("C9.2", L, "First-order Taylor coefficient of the exact RG ratio reproduces it exactly",
           "Eq delta-RG-1 vs delta-GR",
           sp.simplify(d1 - delta_GR),
           f"delta_RG^(1) = {sp.sstr(d1)}",
           extra_syms={kE2, bE2})

    d2_paper = (kE2**2 / 8 + bE2**2 / 8 - bE2 * kE2 / 4
                + 3 * kE2**2 * ratio / 8 - 19 * kE2**2 * ratio**2 / 32
                + 3 * bE2 * kE2 * ratio / 8)
    record("C9.3", L, "Second-order coefficient matches the published 6-term expression",
           "Eq delta-RG-2",
           sp.simplify(d2 - d2_paper),
           f"delta_RG^(2) = {sp.sstr(sp.nsimplify(d2))}",
           extra_syms={kE2, bE2})

    record("C9.4", L, "The beta^2 kappa^2 cross term is absent from any additive GR rearrangement",
           "Interpretation (iii)",
           sp.expand(delta_GR).coeff(kE2 * bE2) - 0,
           "delta_GR is linear in each amplitude; the cross term is purely multiplicative.",
           extra_syms={kE2, bE2})

    # ---- numerical reproduction at 40 digits ------------------------------
    mp = sp.mpmath if hasattr(sp, "mpmath") else __import__("mpmath")
    mp.mp.dps = 40
    c_n = mp.mpf("299792458")
    GM = mp.mpf("3.986004418e14")
    RE = mp.mpf("6378137")
    rG = mp.mpf("26561750")
    Dn = mp.mpf("86400")
    Mn = mp.mpf("1e6")

    kE2n = 2 * GM / (RE * c_n**2)
    kG2n = 2 * GM / (rG * c_n**2)
    vE = 2 * mp.pi * RE / Dn
    bE2n = (vE / c_n) ** 2
    bG2n = GM / (rG * c_n**2)

    tauE = mp.sqrt(1 - kE2n) * mp.sqrt(1 - bE2n)
    tauG = mp.sqrt(1 - kG2n) * mp.sqrt(1 - bG2n)
    dt_RG = (1 - tauE / tauG) * Dn * Mn
    dt_GR = ((-kG2n / 2 + kE2n / 2) - (bG2n - bE2n) / 2) * Dn * Mn

    d2_num = mp.mpf(str(sp.N(d2.subs({kE2: sp.Float(str(kE2n), 40),
                                      bE2: sp.Float(str(bE2n), 40),
                                      ratio: sp.Float(str(RE / rG), 40)}), 40)))
    GPS.update(
        dt_RG=mp.nstr(dt_RG, 12),
        dt_GR=mp.nstr(dt_GR, 12),
        difference=mp.nstr(dt_RG - dt_GR, 6),
        predicted_second_order=mp.nstr(d2_num * Dn * Mn, 6),
        residual=mp.nstr(abs((dt_RG - dt_GR) - d2_num * Dn * Mn), 3),
        closure_residual=mp.nstr(abs(bG2n - kG2n / 2), 3),
        relative_size_of_omitted_term=mp.nstr((dt_RG - dt_GR) / dt_RG, 4),
    )

    rel_err = abs((dt_RG - dt_GR) - d2_num * Dn * Mn) / abs(dt_RG - dt_GR)
    record("C9.5", L, "Numerical: Delta t_RG - Delta t_GR equals the predicted second-order term",
           "Results table (numerical)",
           sp.Integer(0) if rel_err < mp.mpf("1e-6") else sp.Integer(1),
           f"Delta t_RG = {GPS['dt_RG']} us/day, Delta t_GR = {GPS['dt_GR']} us/day, "
           f"difference = {GPS['difference']}, predicted = {GPS['predicted_second_order']}, "
           f"relative agreement = {mp.nstr(rel_err, 3)}")

    record("C9.6", L, "Numerical: circular closure residual beta_GPS^2 - kappa_GPS^2/2 vanishes",
           "Methodology, Part II",
           sp.Integer(0) if abs(bG2n - kG2n / 2) < mp.mpf("1e-45") else sp.Integer(1),
           f"closure residual = {GPS['closure_residual']}")


# ============================================================================
# Zero-parameter audit
# ============================================================================

CONSTANT_PROVENANCE = {
    "1": "unit normalisation of the relational ledger on each carrier",
    "2": "dim S^2 / dim S^1 = DOF count (Closure Theorem); also R_s = 2GM/c^2 bookkeeping",
    "1/2": "first Taylor coefficient of sqrt(1-x) -- origin of the classical 1/2",
    "3": "volumetric proxy exponent r^3, fixed by r-independence of the mass label",
    "4": "square of the DOF ratio entering the tau inversion (1+8 tau^2 quadratic)",
    "8": "8 pi G/c^4 legacy coupling; and the 8 in 1+8 tau^2 from the closure quadratic",
    "1/4": "root of the closure quadratic beta^2 = (3 - sqrt(1+8 tau^2))/4",
    "4*pi": "surface measure of the S^2 carrier (NOT Newton's 4pi/3 volume measure)",
    "8*pi": "2 x 4 pi: carrier surface measure times the R_s = 2GM/c^2 factor",
    "19/32, 3/8": "higher Taylor coefficients of the exact RG ratio (derived, not fitted)",
}


def audit_zero_parameters():
    """A parameter is 'free' iff it could be tuned to data. Verify none exists."""
    rows = []
    for rec in RESULTS:
        rows.append(dict(id=rec["id"], layer=rec["layer"],
                         registers=rec["registers"], smuggled=rec["smuggled"],
                         constants=rec["constants"]))
    smuggled_total = sorted({s for rec in RESULTS for s in rec["smuggled"].split() if s})
    all_constants = sorted({q for rec in RESULTS for q in rec["constants"].split() if q})
    return rows, smuggled_total, all_constants


# Items that are *declared* rather than *derived* inside WILL_RG_I. None of them is
# a fitted parameter, but each is a place where the chain rests on a stipulation.
ASSUMPTION_LEDGER = [
    dict(tag="A1", kind="convention", where="Thm carriers",
         item="Ledger on each carrier normalised to unity (unit radius).",
         status="Verified as the unique parameter-free choice: any radius R != 1 leaves the "
                "residual 1-R^2 (negative control C1.5)."),
    dict(tag="A2", kind="stipulation", where="Lem DOF-Indifference; Thm Restoring Closure",
         item="Independent channels add in quadrature: kappa_tot^2 = sum_i kappa_i^2.",
         status="Asserted from isotropy + minimalism, not independently derived. Load-bearing "
                "for the factor 2 (C3.1) and for the cord's share (C7.6)."),
    dict(tag="A3", kind="translation", where="Box Cross-Cultural Invariants",
         item="beta = v/c and kappa = v_e/c = sqrt(R_s/r).",
         status="Presented as translation into legacy vocabulary rather than definition. "
                "Empirically anchored via z_beta, z_kappa (C2.1-C2.2)."),
    dict(tag="A4", kind="branch choice", where="Eq globes-answer",
         item="Root selection in beta^2 = (3 - sqrt(1+8 tau^2))/4.",
         status="The (3 + sqrt) root gives beta^2 = 1 and is discarded on physical grounds. "
                "Both roots solve the quadratic (C7.2)."),
    dict(tag="A5", kind="internal boundary", where="Sec no_singularities vs Sec potential",
         item="r_min = R_s/2 requires kappa_max^2 = 2, but additive S^2 closure caps kappa^2 <= 1.",
         status="Inside Part I the additive closure restricts CLOSED STATIC states to "
                "beta^2 <= 1/2, i.e. r >= R_s (C3.7). kappa^2 = 2 is reachable only after the "
                "promotion to the multiplicative phase tau^2 = beta_Y^2 kappa_X^2 "
                "(deferred to R.O.M. Kerr). r_min = R_s/2 is therefore NOT self-contained in Part I."),
    dict(tag="A6", kind="asserted relation", where="Sec pressure",
         item="Radial balance P = (c^4/8 pi G)(1/r) d(kappa^2)/dr.",
         status="Stated without derivation in Part I. Given it, P = -rho c^2 follows "
                "identically (C6.5)."),
    dict(tag="A7", kind="translation", where="Sec density",
         item="3D volumetric proxy r^3 plus 1/(4 pi) S^2 surface normalisation.",
         status="Explicitly declared as a legacy-translation interface. Self-consistency then "
                "forces n=3, alpha=4 pi (C6.3-C6.4). NOTE: M = 4 pi r^3 rho differs from "
                "Newton's M = (4 pi/3) r^3 rho by a factor 3, so 'rho' here is not numerically "
                "the Newtonian mass density; the identity kappa^2 = rho/rho_max is internally "
                "consistent but convention-dependent."),
    dict(tag="A8", kind="no new content", where="Sec willinvariant",
         item="W_ILL = E T/(M L) = 1.",
         status="Reduces identically to 2 beta^2/kappa^2 (C8.1), so W_ILL = 1 IS the closure "
                "theorem in dimensionful dress. The phase-normalised form additionally needs "
                "r_o = c t_o (C8.3). No independent predictive content."),
    dict(tag="A9", kind="scope", where="Sec earth-gps",
         item="GPS comparison target is the ADDITIVE 1PN formula.",
         status="Acknowledged in the paper. Not a comparison against full geodesic integration "
                "in exact Schwarzschild plus SR kinematics."),
    dict(tag="A10", kind="no new content", where="Thm Energy Symmetry",
         item="Delta E_{A->B} + Delta E_{B->A} = 0.",
         status="Holds for an arbitrary state function (control C4.10b). The law alone is "
                "vacuous; its content is entirely in E = E_0 kappa_X/beta_Y."),
]

DERIVED_COROLLARIES = [
    "Closed static states satisfy beta^2 <= 1/2, i.e. orbital speed <= c/sqrt(2) and r >= R_s "
    "(C3.7) -- a sharper bound than r >= R_s/2 within Part I's additive closure.",
    "The virial coefficient is not an independent fact: |V| = 2T is literally kappa^2 = 2 beta^2 "
    "rewritten in legacy units (C3.9).",
    "Eccentricity and the closure factor are the same quantity: e = 1/delta - 1 (C3.5).",
    "W_ILL = 1 and kappa^2 = 2 beta^2 are the same statement (C8.1, C8.2).",
    "The GR/RG difference is controlled by the multiplicative cross term -beta^2 kappa^2/4, "
    "which no rearrangement of an additive 1PN sum can produce (C5.2, C9.3, C9.4).",
]


def run_all():
    RESULTS.clear()
    for fn in (layer1, layer2, layer3, layer4, layer5,
               layer6, layer7, layer8, layer9):
        fn()
    return RESULTS


if __name__ == "__main__":
    run_all()
    bad = [r for r in RESULTS if r["verdict"] != "OK"]
    print(f"checks={len(RESULTS)} ok={len(RESULTS) - len(bad)} problems={len(bad)}")
    for r in bad:
        print(f"  {r['id']}: {r['claim']}\n    residual={r['residual']}")

checks=68 ok=68 problems=0
